In [1]:
import torch
import einops

In [2]:
cuda_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# episode_hstrace_indices (memory_indices)

**Дано:**

1) max_episode_steps_count - максимальное число шагов на эпизод. Далее игра заканчивается
2) memory_length - сколько агент может помнить шагов.
3) memory_length <= max_episode_steps_count. Агент помнит не всю глубину игры, а только memory_length.
4) episode_hstraces - хоронилище (в виде тензора-гиперкуба) скрытых состояния агента (max_episode_steps_count, layers_count, d_model) на период длительности эпизода (т.е. пока игра не кончится).

**Задача:**

Для каждого шага в среде нужно получить индексы внутри hidden_states, которые будут играть роль исторического контекста длиной memory_length. Логика формирования индексов показана на схеме. Тут важно, что пока step < memory_length, то мы используем полный memory_length, т.к. там потом всё равно будет использоваться causal mask.

<img src="./img/memory_indices.jpg">

In [95]:
max_episode_steps_count = 300
memory_length = 100

repetitions = torch.repeat_interleave(
    torch.arange(0, memory_length).unsqueeze(0), 
    memory_length - 1, 
    dim=0
).long() # [memory_length-1, memory_length], e.g. [99, 100], where each row=[0, 1, ... 99]
memory_indices_1 = torch.stack(
    [torch.arange(i, i + memory_length) for i in range(max_episode_steps_count - memory_length + 1)]
).long() # [max_episode_steps_count - memory_length + 1, memory_length], e.g. [201, 100]
memory_indices = torch.cat((repetitions, memory_indices_1))

In [96]:
repetitions.shape

torch.Size([99, 100])

In [4]:
repetitions

tensor([[ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99],
        ...,
        [ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99],
        [ 0,  1,  2,  ..., 97, 98, 99]])

In [5]:
memory_indices_1.shape

torch.Size([201, 100])

In [6]:
memory_indices_1

tensor([[  0,   1,   2,  ...,  97,  98,  99],
        [  1,   2,   3,  ...,  98,  99, 100],
        [  2,   3,   4,  ...,  99, 100, 101],
        ...,
        [198, 199, 200,  ..., 295, 296, 297],
        [199, 200, 201,  ..., 296, 297, 298],
        [200, 201, 202,  ..., 297, 298, 299]])

In [7]:
memory_indices.shape

torch.Size([300, 100])

In [8]:
memory_indices

tensor([[  0,   1,   2,  ...,  97,  98,  99],
        [  0,   1,   2,  ...,  97,  98,  99],
        [  0,   1,   2,  ...,  97,  98,  99],
        ...,
        [198, 199, 200,  ..., 295, 296, 297],
        [199, 200, 201,  ..., 296, 297, 298],
        [200, 201, 202,  ..., 297, 298, 299]])

In [103]:
assert torch.all(repetitions[98] == memory_indices_1[0])

In [9]:
# Equivalent code to get indices
for step in range(max_episode_steps_count):
    offset = max(step + 1 - memory_length, 0)
    indices = torch.arange(memory_length) + offset
    assert torch.all(indices == memory_indices[step])

# batched_index_select

**Дано:**
1) global_memory. Для каждой среды (envs_count) выделяем память на max_episode_steps_count. В этой памяти будем хранить, например, скрытые состояния слоёв трансформера (layers_count, d_model)
2) index. Для каждой среды описывает какие шаги составляют контекст (исторические данные) для трансформера. Контекст ограничен memory_length

Т.к. max_episode_steps_count >= memory_length, то это значит, что контекст "смотрит" назад не на всю глубину, а только на memory_length. 

Далее, т.к. в каждой среде играет свой агент, то разные агенты могут по разному бежать. Это значит, что в index для каждого окружения будет свой набор индексов.

**Задача:**

Выбрать из global_memory для каждой среды свой исторический контекст => сформировать единый тензор исторического контекста.

<img src="./img/batched_index_select.jpg" width=800>

**Проблематика:**

Сделать нужно максимально эффективно, без циклов на питоне

In [10]:
envs_count = 4
max_episode_steps_count = 200
layers_count = 3
d_model = 384

global_memory = torch.rand((envs_count, max_episode_steps_count, layers_count, d_model))

In [11]:
memory_length = 10

index = torch.vstack([
    torch.arange(memory_length) + 0,
    torch.arange(memory_length) + 1,
    torch.arange(memory_length) + 2,
    torch.arange(memory_length) + 3,
])

In [12]:
def batched_index_select_orig(input, dim, index):
    for ii in range(1, len(input.shape)):
        if ii != dim:
            index = index.unsqueeze(ii)

    expanse = list(input.shape)
    expanse[0] = -1
    expanse[dim] = -1
    index = index.expand(expanse)
    return torch.gather(input, dim, index)

In [34]:
def batched_index_select(input, dim, index):
    index = index.reshape((*index.shape, 1, 1)) # e.g. [e,s] -> [e,s,1,1]
    index = index.expand((-1, -1, input.shape[2], input.shape[3])) # e.g. [e,s,1,1] -> [e,s,3,384]
    return torch.gather(input, dim, index)

In [14]:
%%timeit
context0 = batched_index_select_orig(global_memory, 1, index)

88.7 μs ± 3.15 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [15]:
%%timeit
context1 = batched_index_select(global_memory, 1, index)

84.4 μs ± 386 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [16]:
%%timeit
assert index.shape[1] == memory_length
context2 = torch.zeros((envs_count, memory_length, layers_count, d_model))

for i in range(envs_count):
    for j in range(memory_length):
        step = index[i,j]
        context2[i,j] = global_memory[i,step]

604 μs ± 1.43 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
context0 = batched_index_select_orig(global_memory, 1, index)
context1 = batched_index_select(global_memory, 1, index)
context2 = torch.zeros((envs_count, memory_length, layers_count, d_model))

for i in range(envs_count):
    for j in range(memory_length):
        step = index[i,j]
        context2[i,j] = global_memory[i,step]

assert context0.shape == context1.shape
assert context1.shape == context2.shape
assert torch.all(context0 == context1)
assert torch.all(context1 == context2)

**ВЫВОД:**

Короче видно, что если написать на цикле, выйдет в 10 раз медленее, чем через хитроумный `batched_index_select`

# episode_hstraces[:,episode_steps] VS episode_hstraces[env_inds,episode_steps]

In [18]:
m = torch.zeros((3,5))
ii = torch.arange(3)
jj = torch.tensor([0,1,3]).long()
m[ii,jj] = torch.ones(3)
m

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0.]])

In [19]:
m = torch.zeros((3,5))
jj = torch.tensor([0,1,3]).long()
m[:,jj] = torch.ones(3)
m

tensor([[1., 1., 0., 1., 0.],
        [1., 1., 0., 1., 0.],
        [1., 1., 0., 1., 0.]])

# unfold vs batched_index_select

In [79]:
envs_count = 32
max_episode_steps_count = 1000
layers_count = 3
d_model = 384

hstraces = torch.rand((envs_count, max_episode_steps_count, layers_count, d_model)).to(cuda_device)

print(torch.cuda.memory_allocated())

155570176


In [88]:
memory_length = 100

index = torch.vstack([
    torch.arange(memory_length) + i for i in range(envs_count)
]).to(cuda_device)

env_inds = torch.arange(len(hstraces)).to(cuda_device)
mem_window_inds = index[:,0].contiguous()

print(torch.cuda.memory_allocated())

158178304


In [89]:
hstrace_windows = hstraces.unfold(dimension=1, size=memory_length, step=1)
hstrace_windows = einops.rearrange(hstrace_windows, 'e w l d m -> e w m l d')
print(f'{hstrace_windows.shape=}')

print(torch.cuda.memory_allocated()) # Ensure that no extra memory is allocated

hstrace_windows.shape=torch.Size([32, 901, 100, 3, 384])
158178304


In [90]:
context0 = batched_index_select(hstraces, 1, index)
context1 = hstrace_windows[env_inds,mem_window_inds]
assert torch.all(context0 == context1)

In [91]:
%%timeit
context0 = batched_index_select(hstraces, 1, index)

28.1 μs ± 336 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [92]:
%%timeit
context1 = hstrace_windows[env_inds,mem_window_inds]

22.4 μs ± 468 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
